# Challenge — Module 2.1: Multi-Format Product Feed

**Format:** one problem, one final artifact. You must touch **every** format from 2.1: `csv` (DictReader), `json` (regular and JSON Lines), `parquet` via `pyarrow`, and `gzip` for compressed input. No `pandas` in this challenge — that comes later. Stick to the stdlib + `pyarrow`.

**Scenario.** Three upstream systems drop product data into a folder, each in a different format. You have to consolidate them into a single Parquet file that downstream pandas/SQL jobs can consume.

## The problem

The setup cell below writes three input files to `feed_in/`:

1. `products.csv.gz` — a gzipped CSV with columns `sku,name,base_price` (price as string).
2. `categories.json` — a regular JSON object mapping `sku -> category`.
3. `inventory.jsonl` — JSON Lines, one record per line: `{"sku": "...", "in_stock": 42}`.

### Your task

Write code that produces `feed_out/products.parquet` with the following schema and rules:

| column      | type    | source                                                |
|-------------|---------|-------------------------------------------------------|
| `sku`       | string  | from CSV                                              |
| `name`      | string  | from CSV                                              |
| `price`     | float64 | cast from CSV `base_price`                            |
| `category`  | string  | looked up from `categories.json` (missing -> `"misc"`)|
| `in_stock`  | int64   | from `inventory.jsonl` (missing -> `0`)               |

Then **read the Parquet file back** and print:

- The number of rows.
- The total inventory value: `sum(price * in_stock)` rounded to 2 decimals.
- The set of distinct categories, sorted alphabetically.

### Expected output (shape)

```
rows=5
total_inventory_value=1234.56
categories=['books', 'electronics', 'misc']
```

(Numbers will match what the setup writes — don't hardcode them.)

### Restrictions

- Read the gzipped CSV **without** decompressing it to disk first — `gzip.open` + `csv.DictReader`.
- For `inventory.jsonl`, parse line-by-line, **not** as one big JSON array.
- Write Parquet via `pyarrow.parquet.write_table` (build an Arrow table from a dict of columns).
- The lookup from `categories.json` should use `.get(sku, 'misc')` — don't crash on missing SKUs.

### Hints

- `gzip.open(path, 'rt', encoding='utf-8')` gives you a text-mode handle that `csv.DictReader` can consume directly.
- Build a list of dicts as you go, then transpose into columns (one list per field) for `pyarrow.Table.from_pydict`.
- `pyarrow` types: `pa.string()`, `pa.float64()`, `pa.int64()`. Use `pa.schema([...])` if you want to be explicit.

In [ ]:
# --- setup: writes the three input files. Run this once. ---
import csv, gzip, json, os
from pathlib import Path

Path('feed_in').mkdir(exist_ok=True)
Path('feed_out').mkdir(exist_ok=True)

products = [
    {'sku': 'A001', 'name': 'USB Cable',   'base_price': '4.50'},
    {'sku': 'A002', 'name': 'HDMI Cable',  'base_price': '8.00'},
    {'sku': 'B001', 'name': 'Notebook',    'base_price': '3.25'},
    {'sku': 'B002', 'name': 'Pen Set',     'base_price': '6.10'},
    {'sku': 'C001', 'name': 'Mystery Box', 'base_price': '15.00'},
]
categories = {'A001': 'electronics', 'A002': 'electronics', 'B001': 'books', 'B002': 'books'}
inventory  = [
    {'sku': 'A001', 'in_stock': 100},
    {'sku': 'A002', 'in_stock': 40},
    {'sku': 'B001', 'in_stock': 200},
    {'sku': 'B002', 'in_stock': 75},
]

with gzip.open('feed_in/products.csv.gz', 'wt', encoding='utf-8', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['sku', 'name', 'base_price'])
    w.writeheader(); w.writerows(products)

with open('feed_in/categories.json', 'w', encoding='utf-8') as f:
    json.dump(categories, f)

with open('feed_in/inventory.jsonl', 'w', encoding='utf-8') as f:
    for rec in inventory:
        f.write(json.dumps(rec) + '\n')

print('setup done')

setup done


In [ ]:
# your code here
import csv, gzip, json
import pyarrow as pa
import pyarrow.parquet as pq

# 1. read feed_in/products.csv.gz       -> list of dicts
# 2. read feed_in/categories.json       -> dict[sku, category]
# 3. read feed_in/inventory.jsonl       -> dict[sku, in_stock]
# 4. merge into columnar form, write feed_out/products.parquet
# 5. read it back and print rows, total_inventory_value, categories
